<a href="https://colab.research.google.com/github/ARehman007-max/internship/blob/main/Task3_datacleaningandpreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()
#uploaded = files.upload

Saving Messy_Employee_dataset.csv to Messy_Employee_dataset.csv


In [2]:
import pandas as pd


In [4]:
import io
df=pd.read_csv(io.BytesIO(uploaded['Messy_Employee_dataset.csv']))

In [5]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [7]:
# Initial inspection
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nFirst 5 rows:")


Dataset Shape: (1020, 12)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 88.8+ KB

Missing Values:
Employee_ID            0
First_Name             0
Last_Name              0
Age             

In [8]:
# Check missing values in each column
print("Missing values before cleaning:")
print(df.isnull().sum())

# 1. Drop rows with missing Email (critical identifier)
df_clean = df.dropna(subset=['Email'])

# 2. Fill Age with median (less sensitive to outliers than mean)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# 3. Fill Performance_Score with mode (most common category)
df_clean['Performance_Score'] = df_clean['Performance_Score'].fillna(df_clean['Performance_Score'].mode()[0])

# 4. Fill Phone with placeholder for missing
df_clean['Phone'] = df_clean['Phone'].fillna('Unknown')

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

Missing values before cleaning:
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

Missing values after cleaning:
Employee_ID           0
First_Name            0
Last_Name             0
Age                   0
Department_Region     0
Status                0
Join_Date             0
Salary               24
Email                 0
Phone                 0
Performance_Score     0
Remote_Work           0
dtype: int64


In [9]:
# Check for duplicates
print("Duplicates before removal:", df_clean.duplicated().sum())

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Check for duplicates based on Employee_ID only (should be unique)
duplicate_ids = df_clean[df_clean.duplicated(subset=['Employee_ID'], keep=False)]
print("Rows with duplicate Employee_ID:")
print(duplicate_ids)

# If duplicates found based on Employee_ID, keep first occurrence
df_clean = df_clean.drop_duplicates(subset=['Employee_ID'], keep='first')

print("Duplicates after removal:", df_clean.duplicated().sum())

Duplicates before removal: 0
Rows with duplicate Employee_ID:
Empty DataFrame
Columns: [Employee_ID, First_Name, Last_Name, Age, Department_Region, Status, Join_Date, Salary, Email, Phone, Performance_Score, Remote_Work]
Index: []
Duplicates after removal: 0


In [10]:
# Standardize Department_Region column
# First split into Department and Region for better standardization
df_clean[['Department', 'Region']] = df_clean['Department_Region'].str.split('-', expand=True)

# Clean Department names (remove extra spaces, standardize capitalization)
df_clean['Department'] = df_clean['Department'].str.strip().str.title()

# Clean Region names
df_clean['Region'] = df_clean['Region'].str.strip().str.title()

# Clean Status column
df_clean['Status'] = df_clean['Status'].str.strip().str.title()

# Clean Performance_Score
df_clean['Performance_Score'] = df_clean['Performance_Score'].str.strip().str.title()

# Clean names
df_clean['First_Name'] = df_clean['First_Name'].str.strip().str.title()
df_clean['Last_Name'] = df_clean['Last_Name'].str.strip().str.title()

# Clean Email (convert to lowercase)
df_clean['Email'] = df_clean['Email'].str.strip().str.lower()

# View unique values to check standardization
print("Unique Departments:")
print(df_clean['Department'].unique())
print("\nUnique Regions:")
print(df_clean['Region'].unique())
print("\nUnique Status:")
print(df_clean['Status'].unique())

Unique Departments:
['Devops' 'Finance' 'Admin' 'Cloud Tech' 'Sales' 'Hr']

Unique Regions:
['California' 'Texas' 'Nevada' 'Florida' 'New York' 'Illinois']

Unique Status:
['Active' 'Pending' 'Inactive']


In [11]:
# Fix Join_Date to datetime
df_clean['Join_Date'] = pd.to_datetime(df_clean['Join_Date'], errors='coerce')

# Fix Phone numbers (convert negative numbers to positive, format as string)
def clean_phone(phone):
    if pd.isna(phone) or phone == 'Unknown':
        return 'Unknown'
    # Convert to string and remove decimal point if float
    phone_str = str(phone)
    if phone_str.endswith('.0'):
        phone_str = phone_str[:-2]
    # Remove negative sign
    if phone_str.startswith('-'):
        phone_str = phone_str[1:]
    return phone_str

df_clean['Phone'] = df_clean['Phone'].apply(clean_phone)

# Fix Salary if it's not already numeric
# Check if Salary is string with $ and commas
if df_clean['Salary'].dtype == 'object':
    df_clean['Salary'] = df_clean['Salary'].str.replace('$', '').str.replace(',', '').astype(float)

# Fix Remote_Work to boolean
df_clean['Remote_Work'] = df_clean['Remote_Work'].astype(bool)

# Verify data types
print("Data types after cleaning:")
print(df_clean.dtypes)

Data types after cleaning:
Employee_ID                  object
First_Name                   object
Last_Name                    object
Age                         float64
Department_Region            object
Status                       object
Join_Date            datetime64[ns]
Salary                      float64
Email                        object
Phone                        object
Performance_Score            object
Remote_Work                    bool
Department                   object
Region                       object
dtype: object


In [12]:
# Detect outliers in Age and Salary using IQR method

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]

    print(f"\n{column} Outlier Analysis:")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower Bound: {lower_bound:.2f}")
    print(f"Upper Bound: {upper_bound:.2f}")
    print(f"Number of outliers: {len(outliers)}")
    print(f"Percentage of outliers: {(len(outliers)/len(data))*100:.2f}%")

    return outliers

# Check Age outliers
age_outliers = detect_outliers_iqr(df_clean, 'Age')
print("\nAge Outliers:")
print(age_outliers[['Employee_ID', 'First_Name', 'Last_Name', 'Age']])

# Check Salary outliers
salary_outliers = detect_outliers_iqr(df_clean, 'Salary')
print("\nSalary Outliers:")
print(salary_outliers[['Employee_ID', 'First_Name', 'Last_Name', 'Salary']])

# Option 1: Remove outliers (use with caution)
# df_clean_no_outliers = df_clean[
#     (df_clean['Age'] >= df_clean['Age'].quantile(0.25) - 1.5 * IQR_age) &
#     (df_clean['Age'] <= df_clean['Age'].quantile(0.75) + 1.5 * IQR_age) &
#     (df_clean['Salary'] >= df_clean['Salary'].quantile(0.25) - 1.5 * IQR_salary) &
#     (df_clean['Salary'] <= df_clean['Salary'].quantile(0.75) + 1.5 * IQR_salary)
# ]

# Option 2: Cap outliers at bounds (winsorization)
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    data[column] = data[column].clip(lower_bound, upper_bound)
    return data

# Apply capping to Age and Salary
df_clean = cap_outliers(df_clean, 'Age')
df_clean = cap_outliers(df_clean, 'Salary')


Age Outlier Analysis:
Q1: 30.00
Q3: 35.00
IQR: 5.00
Lower Bound: 22.50
Upper Bound: 42.50
Number of outliers: 0
Percentage of outliers: 0.00%

Age Outliers:
Empty DataFrame
Columns: [Employee_ID, First_Name, Last_Name, Age]
Index: []

Salary Outlier Analysis:
Q1: 68392.49
Q3: 100974.03
IQR: 32581.54
Lower Bound: 19520.18
Upper Bound: 149846.34
Number of outliers: 0
Percentage of outliers: 0.00%

Salary Outliers:
Empty DataFrame
Columns: [Employee_ID, First_Name, Last_Name, Salary]
Index: []


In [13]:
# Final dataset verification
print("Final Dataset Summary:")
print("="*50)
print(f"Shape: {df_clean.shape}")
print(f"\nMissing Values:\n{df_clean.isnull().sum()}")
print(f"\nDuplicate Rows: {df_clean.duplicated().sum()}")
print(f"\nData Types:\n{df_clean.dtypes}")
print(f"\nBasic Statistics:\n{df_clean.describe()}")

# Drop the original Department_Region column if no longer needed
df_clean = df_clean.drop('Department_Region', axis=1)

# Reorder columns for better organization
column_order = ['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department',
                'Region', 'Status', 'Join_Date', 'Salary', 'Email', 'Phone',
                'Performance_Score', 'Remote_Work']
df_clean = df_clean[column_order]

# Display final cleaned dataset
print("\nFinal Cleaned Dataset (first 10 rows):")
print(df_clean.head(10))

# Save cleaned dataset
df_clean.to_csv('Cleaned_Employee_Dataset.csv', index=False)
print("\nCleaned dataset saved as 'Cleaned_Employee_Dataset.csv'")

Final Dataset Summary:
Shape: (1020, 14)

Missing Values:
Employee_ID           0
First_Name            0
Last_Name             0
Age                   0
Department_Region     0
Status                0
Join_Date             0
Salary               24
Email                 0
Phone                 0
Performance_Score     0
Remote_Work           0
Department            0
Region                0
dtype: int64

Duplicate Rows: 0

Data Types:
Employee_ID                  object
First_Name                   object
Last_Name                    object
Age                         float64
Department_Region            object
Status                       object
Join_Date            datetime64[ns]
Salary                      float64
Email                        object
Phone                        object
Performance_Score            object
Remote_Work                    bool
Department                   object
Region                       object
dtype: object

Basic Statistics:
               Age      